# Prompt Tool Delta Embedding Model Comparison ToolBench Mismatch Ratio Study Experiment

This notebook runs a class-ratio sensitivity study for the delta-embedding detector.

Base data: `toolbench_mismatch_dataset.csv`

Experiment design:

- Keep label 0 normal samples fixed at 100,000 rows.
- Reduce label 1 mismatch samples according to Normal:Mismatch ratios: 1:1, 7:3, 8:2, 10:1, 100:1.
- Use the same delta feature: `prompt_embedding - tool_embedding`.
- Use the same model set: LightGBM, XGBoost, Random Forest, Isolation Forest.
- Use `source_prompt_hash` as the group split key to avoid prompt-level leakage.
- Report Precision, Recall, F1-score, Accuracy, AUROC, AUPRC, confusion matrix, and latency.

Note: the 7:3 case uses 42,857 label 1 rows because 100,000 * 3 / 7 is not an integer.


## Cell 1. Install packages


In [ ]:
%pip install pandas numpy scikit-learn matplotlib lightgbm sentence-transformers torch xgboost


## Cell 2. Imports and experiment settings


In [ ]:
from datetime import datetime
from pathlib import Path
import json
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score
from sklearn.metrics import precision_recall_curve, precision_score, recall_score, roc_auc_score
from sklearn.metrics import roc_curve, silhouette_score
from sklearn.model_selection import GroupShuffleSplit
from xgboost import XGBClassifier

warnings.filterwarnings('ignore', category=ConvergenceWarning)

def find_repository_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'notebooks').exists() and (candidate / 'README.md').exists():
            return candidate
    return cwd.parent if cwd.name == 'notebooks' else cwd


def find_data_asset_root(repo_root: Path) -> Path:
    candidates = [
        repo_root,
        repo_root / 'paper-data-v1.0',
        repo_root / 'release_assets' / 'paper-data-v1.0',
        repo_root / 'data_asset',
    ]
    expected = Path('data') / 'datasets' / 'toolbench_mismatch_dataset.csv'
    for candidate in candidates:
        if (candidate / expected).exists():
            return candidate
    searched = '\n'.join(str(candidate / expected) for candidate in candidates)
    raise FileNotFoundError(
        'Could not find toolbench_mismatch_dataset.csv. '        'Download paper-data-v1.0.zip from GitHub Releases and extract it at the repository root. '        'Windows Explorer may create paper-data-v1.0/; that layout is also supported.\n'
        f'Searched paths:\n{searched}'
    )


ROOT = find_repository_root()
DATA_ASSET_ROOT = find_data_asset_root(ROOT)
DATA_PATH = DATA_ASSET_ROOT / 'data' / 'datasets' / 'toolbench_mismatch_dataset.csv'

RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ROOT = ROOT / 'outputs' / 'runs' / f'toolbench_mismatch_ratio_study_cpu_models_{RUN_TIMESTAMP}'
RUN_ROOT.mkdir(parents=True, exist_ok=True)

TRAIN_RATIO = 0.64
VALID_RATIO = 0.16
TEST_RATIO = 0.20
SPLIT_GROUP_COL = 'source_prompt_hash'
RANDOM_SEED = 42
DEFAULT_THRESHOLDS = np.round(np.arange(0.0, 1.01, 0.01), 2)
LATENCY_SAMPLE_SIZE = 100
RUN_LATENCY_BENCHMARK = True

NORMAL_SAMPLE_SIZE = 100_000
RATIO_CONFIGS = [
    {'ratio_name': '1_1', 'ratio_label': '1:1', 'normal_parts': 1, 'negative_parts': 1},
    {'ratio_name': '7_3', 'ratio_label': '7:3', 'normal_parts': 7, 'negative_parts': 3},
    {'ratio_name': '8_2', 'ratio_label': '8:2', 'normal_parts': 8, 'negative_parts': 2},
    {'ratio_name': '10_1', 'ratio_label': '10:1', 'normal_parts': 10, 'negative_parts': 1},
    {'ratio_name': '100_1', 'ratio_label': '100:1', 'normal_parts': 100, 'negative_parts': 1},
]
MAX_NEGATIVE_SAMPLE_SIZE = max(
    int(round(NORMAL_SAMPLE_SIZE * cfg['negative_parts'] / cfg['normal_parts']))
    for cfg in RATIO_CONFIGS
)

ANALYSIS_SAMPLE_PER_LABEL = 10_000
SILHOUETTE_SAMPLE_PER_LABEL = 1_000

EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
EMBEDDING_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EMBEDDING_BATCH_SIZE = 256 if EMBEDDING_DEVICE == 'cuda' else 64

PREFER_GPU_LIGHTGBM = False
LIGHTGBM_DEVICE_CANDIDATES = ['cpu']
XGBOOST_DEVICE_CANDIDATES = ['cpu']

MODEL_ORDER = [
    'lightgbm',
    'xgboost',
    'random_forest',
    'isolation_forest',
]

plt.style.use('default')

print('Dataset path         :', DATA_PATH)
print('Run root             :', RUN_ROOT)
print('Split ratio          :', f'train={TRAIN_RATIO:.2f}, valid={VALID_RATIO:.2f}, test={TEST_RATIO:.2f}')
print('Split group column   :', SPLIT_GROUP_COL)
print('Normal sample size   :', NORMAL_SAMPLE_SIZE)
print('Max negative size    :', MAX_NEGATIVE_SAMPLE_SIZE)
print('Ratio configs        :', [(cfg['ratio_label'], int(round(NORMAL_SAMPLE_SIZE * cfg['negative_parts'] / cfg['normal_parts']))) for cfg in RATIO_CONFIGS])
print('Embedding model      :', EMBEDDING_MODEL_NAME)
print('Embedding device     :', EMBEDDING_DEVICE)
if EMBEDDING_DEVICE == 'cuda':
    print('Embedding GPU name   :', torch.cuda.get_device_name(0))
else:
    print('Embedding GPU name   :', 'CUDA not available -> CPU fallback')
print('Embedding batch size :', EMBEDDING_BATCH_SIZE)
print('LightGBM devices     :', LIGHTGBM_DEVICE_CANDIDATES)
print('XGBoost devices      :', XGBOOST_DEVICE_CANDIDATES)
print('Model order          :', MODEL_ORDER)
print('Feature design       :', 'delta embedding only (prompt_emb - tool_emb)')


## Cell 3. Shared helper functions


In [ ]:
_EMBEDDING_MODEL = None


def normalize_text(value):
    if pd.isna(value):
        return ''
    return ' '.join(str(value).strip().split())


def get_embedding_model():
    global _EMBEDDING_MODEL
    if _EMBEDDING_MODEL is None:
        print(f'Loading embedding model: {EMBEDDING_MODEL_NAME} on {EMBEDDING_DEVICE}')
        _EMBEDDING_MODEL = SentenceTransformer(EMBEDDING_MODEL_NAME, device=EMBEDDING_DEVICE)
    return _EMBEDDING_MODEL


def synchronize_if_needed():
    if EMBEDDING_DEVICE == 'cuda':
        torch.cuda.synchronize()


def encode_unique_text_column(series, prefix, model):
    prepared = series.fillna('').astype(str).map(normalize_text)
    codes, uniques = pd.factorize(prepared, sort=False)
    unique_texts = uniques.tolist()
    show_progress = len(unique_texts) > 1000

    print(f'Encoding {prefix}: unique texts={len(unique_texts):,}')
    unique_embeddings = model.encode(
        unique_texts,
        batch_size=EMBEDDING_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=show_progress,
    )
    unique_embeddings = np.asarray(unique_embeddings, dtype=np.float32)
    row_embeddings = unique_embeddings[codes]

    feature_cols = [f'{prefix}_emb_{index:03d}' for index in range(row_embeddings.shape[1])]
    feature_frame = pd.DataFrame(row_embeddings, columns=feature_cols, index=series.index, dtype=np.float32)
    return feature_frame, feature_cols


def build_delta_feature_frame(df):
    metadata_cols = [
        col
        for col in [
            'prompt_source_file',
            'prompt_source_toolkit',
            'tool_source_file',
            'tool_source_toolkit',
            'source_prompt_hash',
            'normal_tool_call_text',
            'mismatch_sampling_type',
        ]
        if col in df.columns
    ]
    out = df[['user_prompt', 'tool_call_text', 'label'] + metadata_cols].copy()
    out['user_prompt'] = out['user_prompt'].fillna('').astype(str).map(normalize_text)
    out['tool_call_text'] = out['tool_call_text'].fillna('').astype(str).map(normalize_text)
    out['label'] = out['label'].astype(int)

    model = get_embedding_model()
    prompt_frame, prompt_feature_cols = encode_unique_text_column(out['user_prompt'], 'prompt', model)
    tool_frame, tool_feature_cols = encode_unique_text_column(out['tool_call_text'], 'tool', model)

    prompt_matrix = prompt_frame.to_numpy(dtype=np.float32)
    tool_matrix = tool_frame.to_numpy(dtype=np.float32)
    delta_matrix = (prompt_matrix - tool_matrix).astype(np.float32)

    delta_feature_cols = [f'delta_emb_{index:03d}' for index in range(delta_matrix.shape[1])]
    delta_frame = pd.DataFrame(delta_matrix, columns=delta_feature_cols, index=out.index, dtype=np.float32)

    out = pd.concat([out, delta_frame], axis=1)
    out[delta_feature_cols] = out[delta_feature_cols].fillna(0)
    return out, delta_feature_cols



def group_train_valid_test_split(
    df,
    group_col=SPLIT_GROUP_COL,
    label_col='label',
    train_ratio=TRAIN_RATIO,
    valid_ratio=VALID_RATIO,
    test_ratio=TEST_RATIO,
    random_seed=RANDOM_SEED,
    max_attempts=30,
):
    if not np.isclose(train_ratio + valid_ratio + test_ratio, 1.0):
        raise ValueError('train_ratio + valid_ratio + test_ratio must sum to 1.0')

    groups = df[group_col].astype(str)
    inner_valid_ratio = valid_ratio / (train_ratio + valid_ratio)

    for offset in range(max_attempts):
        split_seed = random_seed + offset
        outer_splitter = GroupShuffleSplit(n_splits=1, test_size=test_ratio, random_state=split_seed)
        train_valid_idx, test_idx = next(outer_splitter.split(df, y=df[label_col], groups=groups))

        train_valid_df = df.iloc[train_valid_idx].reset_index(drop=True)
        test_df = df.iloc[test_idx].reset_index(drop=True)

        inner_groups = train_valid_df[group_col].astype(str)
        inner_splitter = GroupShuffleSplit(n_splits=1, test_size=inner_valid_ratio, random_state=split_seed)
        train_idx, valid_idx = next(
            inner_splitter.split(train_valid_df, y=train_valid_df[label_col], groups=inner_groups)
        )

        train_df = train_valid_df.iloc[train_idx].reset_index(drop=True)
        valid_df = train_valid_df.iloc[valid_idx].reset_index(drop=True)

        if all(split_df[label_col].nunique() == 2 for split_df in [train_df, valid_df, test_df]):
            return train_df, valid_df, test_df, split_seed

    raise ValueError('Could not create a split where every partition contains both labels.')


def print_split_summary(split_name, df, label_col='label'):
    counts = df[label_col].value_counts().sort_index().to_dict()
    print(f'{split_name:<5} rows={len(df):>8} | label counts={counts}')


def evaluate_thresholds(y_true, y_scores, thresholds):
    y_true = np.asarray(y_true, dtype=int)
    rows = []
    for threshold in thresholds:
        y_pred = (y_scores >= threshold).astype(int)
        rows.append(
            {
                'threshold': float(threshold),
                'precision': float(precision_score(y_true, y_pred, zero_division=0)),
                'recall': float(recall_score(y_true, y_pred, zero_division=0)),
                'f1': float(f1_score(y_true, y_pred, zero_division=0)),
                'accuracy': float(accuracy_score(y_true, y_pred)),
            }
        )

    metrics_df = pd.DataFrame(rows)
    best_row = metrics_df.sort_values(
        by=['f1', 'precision', 'recall', 'accuracy', 'threshold'],
        ascending=[False, False, False, False, True],
    ).iloc[0]
    return metrics_df, best_row


def build_continuous_threshold_grid(scores, num=201):
    scores = np.asarray(scores, dtype=float)
    low = float(np.min(scores))
    high = float(np.max(scores))
    if np.isclose(low, high):
        return np.array([low], dtype=float)
    return np.linspace(low, high, num=num)


def plot_threshold_curve(metrics_df, best_threshold, output_path, title):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(metrics_df['threshold'], metrics_df['f1'], label='F1')
    ax.plot(metrics_df['threshold'], metrics_df['precision'], label='Precision', alpha=0.8)
    ax.plot(metrics_df['threshold'], metrics_df['recall'], label='Recall', alpha=0.8)
    ax.axvline(best_threshold, color='red', linestyle='--', label=f'Best threshold={best_threshold:.4f}')
    ax.set_title(title)
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Score')
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def plot_confusion_matrix_figure(cm, output_path, title):
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    image = ax.imshow(cm, cmap='Blues')
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

    labels = ['Pred 0', 'Pred 1']
    ax.set_xticks([0, 1], labels=labels)
    ax.set_yticks([0, 1], labels=['True 0', 'True 1'])
    ax.set_title(title)

    for row_idx in range(cm.shape[0]):
        for col_idx in range(cm.shape[1]):
            ax.text(col_idx, row_idx, int(cm[row_idx, col_idx]), ha='center', va='center', color='black')

    ax.set_xlabel('Prediction')
    ax.set_ylabel('Ground Truth')
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def plot_score_histogram(score_df, output_path, title):
    fig, ax = plt.subplots(figsize=(8, 4))
    negatives = score_df.loc[score_df['label'] == 0, 'score']
    positives = score_df.loc[score_df['label'] == 1, 'score']
    ax.hist(negatives, bins=30, alpha=0.7, label='Label 0')
    ax.hist(positives, bins=30, alpha=0.7, label='Label 1')
    ax.set_title(title)
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def make_lightgbm_classifiers(scale_pos_weight):
    models = []
    for device_type in LIGHTGBM_DEVICE_CANDIDATES:
        params = {
            'objective': 'binary',
            'n_estimators': 1000,
            'learning_rate': 0.05,
            'num_leaves': 31,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'random_state': RANDOM_SEED,
            'scale_pos_weight': scale_pos_weight,
            'device_type': device_type,
        }
        if device_type == 'gpu':
            params.setdefault('max_bin', 255)
        models.append((LGBMClassifier(**params), device_type))
    return models


def make_xgboost_classifier(scale_pos_weight):
    models = []
    for device_name in XGBOOST_DEVICE_CANDIDATES:
        model = XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=1,
            reg_lambda=1.0,
            random_state=RANDOM_SEED,
            scale_pos_weight=scale_pos_weight,
            tree_method='hist',
            device=device_name,
            n_jobs=-1,
        )
        models.append((model, device_name))
    return models


def train_supervised_model(model_name, train_df, valid_df, feature_cols):
    X_train = train_df[feature_cols].to_numpy(dtype=np.float32)
    y_train = train_df['label'].to_numpy(dtype=int)
    X_valid = valid_df[feature_cols].to_numpy(dtype=np.float32)
    y_valid = valid_df['label'].to_numpy(dtype=int)

    positive_count = int(np.sum(y_train == 1))
    negative_count = int(np.sum(y_train == 0))
    scale_pos_weight = negative_count / max(positive_count, 1)

    if model_name == 'lightgbm':
        last_error = None
        for model, device_type in make_lightgbm_classifiers(scale_pos_weight=scale_pos_weight):
            try:
                fit_start = time.perf_counter()
                model.fit(
                    X_train,
                    y_train,
                    eval_set=[(X_valid, y_valid)],
                    eval_metric='binary_logloss',
                    callbacks=[
                        early_stopping(stopping_rounds=50, verbose=True),
                        log_evaluation(period=50),
                    ],
                )
                fit_seconds = time.perf_counter() - fit_start
                metadata = {'model_device_type': device_type, 'fit_seconds': float(fit_seconds)}
                return model, metadata
            except Exception as exc:
                last_error = exc
                print(f'LightGBM fit failed on device={device_type}: {exc}')
        raise RuntimeError(f'LightGBM fit failed on every device candidate. Last error: {last_error}')

    if model_name == 'xgboost':
        last_error = None
        for model, device_type in make_xgboost_classifier(scale_pos_weight=scale_pos_weight):
            try:
                fit_start = time.perf_counter()
                model.fit(
                    X_train,
                    y_train,
                    eval_set=[(X_valid, y_valid)],
                    verbose=False,
                )
                fit_seconds = time.perf_counter() - fit_start
                metadata = {'model_device_type': device_type, 'fit_seconds': float(fit_seconds)}
                return model, metadata
            except Exception as exc:
                last_error = exc
                print(f'XGBoost fit failed on device={device_type}: {exc}')
        raise RuntimeError(f'XGBoost fit failed on every device candidate. Last error: {last_error}')

    if model_name == 'random_forest':
        model = RandomForestClassifier(
            n_estimators=400,
            class_weight='balanced_subsample',
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )
        fit_start = time.perf_counter()
        model.fit(X_train, y_train)
        fit_seconds = time.perf_counter() - fit_start
        return model, {'model_device_type': 'cpu', 'fit_seconds': float(fit_seconds)}

    raise ValueError(f'Unsupported supervised model: {model_name}')


def build_split_scored_frame(train_df, valid_df, test_df, train_scores, valid_scores, test_scores):
    train_part = train_df[['user_prompt', 'tool_call_text', 'label']].copy()
    train_part['split'] = 'train'
    train_part['score'] = train_scores

    valid_part = valid_df[['user_prompt', 'tool_call_text', 'label']].copy()
    valid_part['split'] = 'valid'
    valid_part['score'] = valid_scores

    test_part = test_df[['user_prompt', 'tool_call_text', 'label']].copy()
    test_part['split'] = 'test'
    test_part['score'] = test_scores

    return pd.concat([train_part, valid_part, test_part], ignore_index=True)


def run_supervised_experiment(model_name, train_df, valid_df, test_df, feature_cols, run_root):
    model_dir = run_root / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    model, model_metadata = train_supervised_model(model_name, train_df, valid_df, feature_cols)

    X_train = train_df[feature_cols].to_numpy(dtype=np.float32)
    y_train = train_df['label'].to_numpy(dtype=int)
    X_valid = valid_df[feature_cols].to_numpy(dtype=np.float32)
    y_valid = valid_df['label'].to_numpy(dtype=int)
    X_test = test_df[feature_cols].to_numpy(dtype=np.float32)
    y_test = test_df['label'].to_numpy(dtype=int)

    train_scores = model.predict_proba(X_train)[:, 1].astype(float)
    valid_scores = model.predict_proba(X_valid)[:, 1].astype(float)
    test_scores = model.predict_proba(X_test)[:, 1].astype(float)

    threshold_metrics_df, best_row = evaluate_thresholds(y_valid, valid_scores, DEFAULT_THRESHOLDS)
    best_threshold = float(best_row['threshold'])
    test_pred = (test_scores >= best_threshold).astype(int)

    precision = float(precision_score(y_test, test_pred, zero_division=0))
    recall = float(recall_score(y_test, test_pred, zero_division=0))
    f1 = float(f1_score(y_test, test_pred, zero_division=0))
    accuracy = float(accuracy_score(y_test, test_pred))
    auroc = float(roc_auc_score(y_test, test_scores))
    auprc = float(average_precision_score(y_test, test_scores))
    cm = confusion_matrix(y_test, test_pred, labels=[0, 1])

    split_scored_df = build_split_scored_frame(train_df, valid_df, test_df, train_scores, valid_scores, test_scores)
    test_predictions_df = test_df[['user_prompt', 'tool_call_text', 'label']].copy()
    test_predictions_df['score'] = test_scores
    test_predictions_df['pred_label'] = test_pred

    threshold_metrics_path = model_dir / 'valid_threshold_metrics.csv'
    split_scored_path = model_dir / 'split_scored.csv'
    test_predictions_path = model_dir / 'test_predictions.csv'
    confusion_path = model_dir / 'test_confusion_matrix.png'
    threshold_curve_path = model_dir / 'valid_f1_curve.png'
    histogram_path = model_dir / 'test_score_histogram.png'
    summary_path = model_dir / 'summary.json'

    threshold_metrics_df.to_csv(threshold_metrics_path, index=False, encoding='utf-8-sig')
    split_scored_df.to_csv(split_scored_path, index=False, encoding='utf-8-sig')
    test_predictions_df.to_csv(test_predictions_path, index=False, encoding='utf-8-sig')

    plot_confusion_matrix_figure(cm, confusion_path, f'Test Confusion Matrix - {model_name}')
    plot_threshold_curve(threshold_metrics_df, best_threshold, threshold_curve_path, f'Validation Threshold Sweep - {model_name}')
    plot_score_histogram(test_predictions_df[['label', 'score']], histogram_path, f'Test Score Histogram - {model_name}')

    summary = {
        'model_name': model_name,
        'score_kind': 'probability',
        'best_validation_threshold': best_threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'accuracy': accuracy,
        'auroc': auroc,
        'auprc': auprc,
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
        'train_rows': int(len(train_df)),
        'valid_rows': int(len(valid_df)),
        'test_rows': int(len(test_df)),
        'model_device_type': model_metadata['model_device_type'],
        'embedding_device': EMBEDDING_DEVICE,
        'fit_seconds': model_metadata['fit_seconds'],
        'output_dir': str(model_dir),
    }
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

    return {
        'model_name': model_name,
        'model': model,
        'feature_cols': feature_cols,
        'score_kind': 'probability',
        'summary': summary,
        'test_predictions': test_predictions_df,
        'split_scored': split_scored_df,
        'output_dir': model_dir,
        'summary_path': summary_path,
    }


def train_isolation_forest(train_df, feature_cols):
    X_train = train_df[feature_cols].to_numpy(dtype=np.float32)
    y_train = train_df['label'].to_numpy(dtype=int)
    X_train_normal = X_train[y_train == 0]

    model = IsolationForest(
        n_estimators=400,
        contamination='auto',
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    fit_start = time.perf_counter()
    model.fit(X_train_normal)
    fit_seconds = time.perf_counter() - fit_start
    return model, {'model_device_type': 'cpu', 'fit_seconds': float(fit_seconds)}


def run_isolation_forest_experiment(train_df, valid_df, test_df, feature_cols, run_root):
    model_name = 'isolation_forest'
    model_dir = run_root / model_name
    model_dir.mkdir(parents=True, exist_ok=True)

    model, model_metadata = train_isolation_forest(train_df, feature_cols)

    X_train = train_df[feature_cols].to_numpy(dtype=np.float32)
    y_train = train_df['label'].to_numpy(dtype=int)
    X_valid = valid_df[feature_cols].to_numpy(dtype=np.float32)
    y_valid = valid_df['label'].to_numpy(dtype=int)
    X_test = test_df[feature_cols].to_numpy(dtype=np.float32)
    y_test = test_df['label'].to_numpy(dtype=int)

    train_scores = (-model.score_samples(X_train)).astype(float)
    valid_scores = (-model.score_samples(X_valid)).astype(float)
    test_scores = (-model.score_samples(X_test)).astype(float)

    threshold_grid = build_continuous_threshold_grid(valid_scores, num=201)
    threshold_metrics_df, best_row = evaluate_thresholds(y_valid, valid_scores, threshold_grid)
    best_threshold = float(best_row['threshold'])
    test_pred = (test_scores >= best_threshold).astype(int)

    precision = float(precision_score(y_test, test_pred, zero_division=0))
    recall = float(recall_score(y_test, test_pred, zero_division=0))
    f1 = float(f1_score(y_test, test_pred, zero_division=0))
    accuracy = float(accuracy_score(y_test, test_pred))
    auroc = float(roc_auc_score(y_test, test_scores))
    auprc = float(average_precision_score(y_test, test_scores))
    cm = confusion_matrix(y_test, test_pred, labels=[0, 1])

    split_scored_df = build_split_scored_frame(train_df, valid_df, test_df, train_scores, valid_scores, test_scores)
    test_predictions_df = test_df[['user_prompt', 'tool_call_text', 'label']].copy()
    test_predictions_df['score'] = test_scores
    test_predictions_df['pred_label'] = test_pred

    threshold_metrics_path = model_dir / 'valid_threshold_metrics.csv'
    split_scored_path = model_dir / 'split_scored.csv'
    test_predictions_path = model_dir / 'test_predictions.csv'
    confusion_path = model_dir / 'test_confusion_matrix.png'
    threshold_curve_path = model_dir / 'valid_f1_curve.png'
    histogram_path = model_dir / 'test_score_histogram.png'
    summary_path = model_dir / 'summary.json'

    threshold_metrics_df.to_csv(threshold_metrics_path, index=False, encoding='utf-8-sig')
    split_scored_df.to_csv(split_scored_path, index=False, encoding='utf-8-sig')
    test_predictions_df.to_csv(test_predictions_path, index=False, encoding='utf-8-sig')

    plot_confusion_matrix_figure(cm, confusion_path, 'Test Confusion Matrix - isolation_forest')
    plot_threshold_curve(threshold_metrics_df, best_threshold, threshold_curve_path, 'Validation Threshold Sweep - isolation_forest')
    plot_score_histogram(test_predictions_df[['label', 'score']], histogram_path, 'Test Score Histogram - isolation_forest')

    summary = {
        'model_name': model_name,
        'score_kind': 'anomaly_score',
        'best_validation_threshold': best_threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'accuracy': accuracy,
        'auroc': auroc,
        'auprc': auprc,
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
        'train_rows': int(len(train_df)),
        'valid_rows': int(len(valid_df)),
        'test_rows': int(len(test_df)),
        'model_device_type': model_metadata['model_device_type'],
        'embedding_device': EMBEDDING_DEVICE,
        'fit_seconds': model_metadata['fit_seconds'],
        'output_dir': str(model_dir),
    }
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

    return {
        'model_name': model_name,
        'model': model,
        'feature_cols': feature_cols,
        'score_kind': 'anomaly_score',
        'summary': summary,
        'test_predictions': test_predictions_df,
        'split_scored': split_scored_df,
        'output_dir': model_dir,
        'summary_path': summary_path,
    }


def build_single_row_feature_frame(user_prompt, tool_call_text, feature_cols):
    model = get_embedding_model()
    prompt_text = normalize_text(user_prompt)
    tool_text = normalize_text(tool_call_text)

    synchronize_if_needed()
    embedding_start = time.perf_counter()
    prompt_emb = model.encode(
        [prompt_text],
        batch_size=1,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32)
    tool_emb = model.encode(
        [tool_text],
        batch_size=1,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32)
    synchronize_if_needed()
    embedding_ms = (time.perf_counter() - embedding_start) * 1000.0

    delta_vec = (prompt_emb[0] - tool_emb[0]).astype(np.float32)
    row_values = {f'delta_emb_{index:03d}': float(value) for index, value in enumerate(delta_vec)}

    feature_frame = pd.DataFrame([{column: row_values[column] for column in feature_cols}], columns=feature_cols)
    feature_frame = feature_frame.astype(np.float32)
    return feature_frame, embedding_ms



def score_single_row(model, score_kind, feature_frame):
    feature_array = feature_frame.to_numpy(dtype=np.float32)
    if score_kind == 'probability':
        return float(model.predict_proba(feature_array)[0, 1])
    if score_kind == 'anomaly_score':
        return float((-model.score_samples(feature_array))[0])
    raise ValueError(f'Unsupported score_kind: {score_kind}')


def predict_single_row_with_latency(user_prompt, tool_call_text, result):
    total_start = time.perf_counter()
    feature_frame, embedding_ms = build_single_row_feature_frame(
        user_prompt=user_prompt,
        tool_call_text=tool_call_text,
        feature_cols=result['feature_cols'],
    )

    model_start = time.perf_counter()
    score = score_single_row(
        model=result['model'],
        score_kind=result['score_kind'],
        feature_frame=feature_frame,
    )
    model_inference_ms = (time.perf_counter() - model_start) * 1000.0
    end_to_end_ms = (time.perf_counter() - total_start) * 1000.0

    pred_label = int(score >= result['summary']['best_validation_threshold'])
    return {
        'pred_score': float(score),
        'pred_label': pred_label,
        'embedding_ms': float(embedding_ms),
        'model_inference_ms': float(model_inference_ms),
        'end_to_end_ms': float(end_to_end_ms),
    }


def benchmark_latency(result, sample_size=LATENCY_SAMPLE_SIZE):
    latency_input_df = (
        result['test_predictions'][['user_prompt', 'tool_call_text', 'label']]
        .sample(n=min(sample_size, len(result['test_predictions'])), random_state=RANDOM_SEED)
        .reset_index(drop=True)
    )

    latency_rows = []
    for row in latency_input_df.itertuples(index=False):
        latency_result = predict_single_row_with_latency(
            user_prompt=row.user_prompt,
            tool_call_text=row.tool_call_text,
            result=result,
        )
        latency_rows.append(
            {
                'user_prompt': row.user_prompt,
                'tool_call_text': row.tool_call_text,
                'true_label': int(row.label),
                **latency_result,
            }
        )

    latency_samples_df = pd.DataFrame(latency_rows)
    latency_samples_path = result['output_dir'] / 'latency_samples.csv'
    latency_summary_path = result['output_dir'] / 'latency_summary.json'

    latency_samples_df.to_csv(latency_samples_path, index=False, encoding='utf-8-sig')

    latency_summary = {
        'sample_size': int(len(latency_samples_df)),
        'embedding_device': EMBEDDING_DEVICE,
        'model_device_type': result['summary']['model_device_type'],
        'embedding_ms_mean': float(latency_samples_df['embedding_ms'].mean()),
        'embedding_ms_median': float(latency_samples_df['embedding_ms'].median()),
        'embedding_ms_p95': float(np.percentile(latency_samples_df['embedding_ms'], 95)),
        'model_inference_ms_mean': float(latency_samples_df['model_inference_ms'].mean()),
        'model_inference_ms_median': float(latency_samples_df['model_inference_ms'].median()),
        'model_inference_ms_p95': float(np.percentile(latency_samples_df['model_inference_ms'], 95)),
        'end_to_end_ms_mean': float(latency_samples_df['end_to_end_ms'].mean()),
        'end_to_end_ms_median': float(latency_samples_df['end_to_end_ms'].median()),
        'end_to_end_ms_p95': float(np.percentile(latency_samples_df['end_to_end_ms'], 95)),
        'latency_samples_path': str(latency_samples_path),
    }
    latency_summary_path.write_text(json.dumps(latency_summary, ensure_ascii=False, indent=2), encoding='utf-8')

    result['summary']['latency_summary_path'] = str(latency_summary_path)
    result['summary']['latency_summary'] = latency_summary

    summary_payload = json.loads(result['summary_path'].read_text(encoding='utf-8'))
    summary_payload['latency_summary_path'] = str(latency_summary_path)
    summary_payload['latency_summary'] = latency_summary
    result['summary_path'].write_text(json.dumps(summary_payload, ensure_ascii=False, indent=2), encoding='utf-8')

    latency_report_df = pd.DataFrame(
        [
            {
                'metric': 'embedding_ms',
                'mean': latency_summary['embedding_ms_mean'],
                'median': latency_summary['embedding_ms_median'],
                'p95': latency_summary['embedding_ms_p95'],
            },
            {
                'metric': 'model_inference_ms',
                'mean': latency_summary['model_inference_ms_mean'],
                'median': latency_summary['model_inference_ms_median'],
                'p95': latency_summary['model_inference_ms_p95'],
            },
            {
                'metric': 'end_to_end_ms',
                'mean': latency_summary['end_to_end_ms_mean'],
                'median': latency_summary['end_to_end_ms_median'],
                'p95': latency_summary['end_to_end_ms_p95'],
            },
        ]
    )
    return latency_samples_df, latency_summary, latency_report_df


## Cell 4. Load the full mismatch dataset


In [ ]:
raw_df = pd.read_csv(DATA_PATH)

required_cols = ['user_prompt', 'tool_call_text', 'label', SPLIT_GROUP_COL]
missing_cols = [col for col in required_cols if col not in raw_df.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

raw_df['label'] = raw_df['label'].astype(int)
label_counts = raw_df['label'].value_counts().sort_index().to_dict()
if label_counts.get(0, 0) < NORMAL_SAMPLE_SIZE:
    raise ValueError(f'Not enough label 0 rows: {label_counts.get(0, 0)} < {NORMAL_SAMPLE_SIZE}')
if label_counts.get(1, 0) < MAX_NEGATIVE_SAMPLE_SIZE:
    raise ValueError(f'Not enough label 1 rows: {label_counts.get(1, 0)} < {MAX_NEGATIVE_SAMPLE_SIZE}')

label0_groups = set(raw_df.loc[raw_df['label'] == 0, SPLIT_GROUP_COL].astype(str))
label1_groups = set(raw_df.loc[raw_df['label'] == 1, SPLIT_GROUP_COL].astype(str))

print('Dataset shape          :', raw_df.shape)
print('Label counts           :', label_counts)
print('Unique split groups    :', raw_df[SPLIT_GROUP_COL].nunique())
print('Label 0 groups         :', len(label0_groups))
print('Label 1 groups         :', len(label1_groups))
print('Groups shared by labels:', len(label0_groups & label1_groups))
print('Columns                :', raw_df.columns.tolist())

preview_cols = [
    col
    for col in [
        'user_prompt',
        'tool_call_text',
        'label',
        'prompt_source_file',
        'prompt_source_toolkit',
        'tool_source_toolkit',
        'source_prompt_hash',
        'normal_tool_call_text',
        'mismatch_sampling_type',
    ]
    if col in raw_df.columns
]
raw_df[preview_cols].head(5)


## Cell 5. Build delta embedding features


In [ ]:
feature_df, feature_cols = build_delta_feature_frame(raw_df)

print('Total feature count  :', len(feature_cols))
print('Delta embedding dims :', len(feature_cols))
print('Feature frame shape  :', feature_df[feature_cols].shape)
print('Feature frame labels :', feature_df['label'].value_counts().sort_index().to_dict())


## Cell 6. Feature list check


In [ ]:
feature_summary_df = pd.DataFrame(
    {
        'feature_name': feature_cols,
        'feature_group': ['delta_embedding' for _ in feature_cols],
    }
)

print('Feature group counts:')
display(feature_summary_df['feature_group'].value_counts().rename_axis('feature_group').reset_index(name='count'))

print('First 20 feature names:')
display(feature_summary_df.head(20))

print('Last 20 feature names:')
display(feature_summary_df.tail(20))


## Cell 7. Sampled delta embedding distribution


In [ ]:
analysis_parts = []
for label_value in [0, 1]:
    subset = feature_df[feature_df['label'] == label_value]
    sample_size = min(ANALYSIS_SAMPLE_PER_LABEL, len(subset))
    analysis_parts.append(subset.sample(n=sample_size, random_state=RANDOM_SEED + label_value))

analysis_df = (
    pd.concat(analysis_parts, ignore_index=True)
    .sample(frac=1, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)
analysis_delta_matrix = analysis_df[feature_cols].to_numpy(dtype=np.float32)
analysis_labels = analysis_df['label'].to_numpy(dtype=int)
analysis_delta_norms = np.linalg.norm(analysis_delta_matrix, axis=1).astype(np.float32)

pca = PCA(n_components=2, random_state=RANDOM_SEED)
delta_pca_2d = pca.fit_transform(analysis_delta_matrix)
plot_df = analysis_df[['label']].copy()
plot_df['delta_norm'] = analysis_delta_norms
plot_df['pca_x'] = delta_pca_2d[:, 0]
plot_df['pca_y'] = delta_pca_2d[:, 1]

pca_plot_path = RUN_ROOT / 'delta_embedding_pca_sampled.png'
norm_plot_path = RUN_ROOT / 'delta_embedding_norm_histogram_sampled.png'

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for label_value, color, label_name in [(0, '#1f77b4', 'Normal (0)'), (1, '#d62728', 'Mismatch (1)')]:
    subset = plot_df[plot_df['label'] == label_value]
    axes[0].scatter(
        subset['pca_x'],
        subset['pca_y'],
        s=12,
        alpha=0.55,
        c=color,
        label=label_name,
    )

axes[0].set_title('Delta embedding PCA (sampled)')
axes[0].set_xlabel('PCA 1')
axes[0].set_ylabel('PCA 2')
axes[0].legend()
axes[0].grid(alpha=0.2)

for label_value, color, label_name in [(0, '#1f77b4', 'Normal (0)'), (1, '#d62728', 'Mismatch (1)')]:
    subset = plot_df[plot_df['label'] == label_value]
    axes[1].hist(
        subset['delta_norm'],
        bins=40,
        alpha=0.55,
        color=color,
        label=label_name,
        density=True,
    )

axes[1].set_title('Delta embedding L2 norm distribution (sampled)')
axes[1].set_xlabel('L2 norm of (prompt_emb - tool_emb)')
axes[1].set_ylabel('Density')
axes[1].legend()
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.savefig(pca_plot_path, dpi=200, bbox_inches='tight')
plt.savefig(norm_plot_path, dpi=200, bbox_inches='tight')
plt.show()

variance_ratio = pca.explained_variance_ratio_
print('Analysis sample shape:', analysis_df.shape)
print('PCA plot path        :', pca_plot_path)
print('Norm plot path       :', norm_plot_path)
print('PCA variance         :', [round(float(v), 4) for v in variance_ratio])
print('Label-wise delta norm summary:')
display(
    plot_df.groupby('label')['delta_norm']
    .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
    .reset_index()
)


## Cell 8. Sampled centroid and projection analysis


In [ ]:
normal_matrix = analysis_delta_matrix[analysis_labels == 0]
negative_matrix = analysis_delta_matrix[analysis_labels == 1]

normal_centroid = normal_matrix.mean(axis=0)
negative_centroid = negative_matrix.mean(axis=0)
centroid_delta = negative_centroid - normal_centroid
centroid_delta_norm = float(np.linalg.norm(centroid_delta))

normal_centroid_norm = float(np.linalg.norm(normal_centroid))
negative_centroid_norm = float(np.linalg.norm(negative_centroid))
centroid_cosine = float(
    np.dot(normal_centroid, negative_centroid)
    / max(normal_centroid_norm * negative_centroid_norm, 1e-12)
)

normal_spread = np.linalg.norm(normal_matrix - normal_centroid, axis=1)
negative_spread = np.linalg.norm(negative_matrix - negative_centroid, axis=1)

if centroid_delta_norm > 0:
    separation_axis = centroid_delta / centroid_delta_norm
else:
    separation_axis = np.zeros_like(centroid_delta)

projection_scores = analysis_delta_matrix @ separation_axis
projection_df = pd.DataFrame(
    {
        'label': analysis_labels,
        'projection_score': projection_scores,
    }
)

projection_plot_path = RUN_ROOT / 'delta_embedding_projection_histogram_sampled.png'
fig, ax = plt.subplots(figsize=(10, 5))
for label_value, color, label_name in [(0, '#1f77b4', 'Normal (0)'), (1, '#d62728', 'Mismatch (1)')]:
    subset = projection_df[projection_df['label'] == label_value]
    ax.hist(
        subset['projection_score'],
        bins=40,
        alpha=0.55,
        density=True,
        color=color,
        label=label_name,
    )
ax.set_title('Projection on centroid-difference axis (sampled)')
ax.set_xlabel('Projection score')
ax.set_ylabel('Density')
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(projection_plot_path, dpi=200, bbox_inches='tight')
plt.show()

silhouette_parts = []
for label_value in [0, 1]:
    subset = analysis_df[analysis_df['label'] == label_value]
    sample_size = min(SILHOUETTE_SAMPLE_PER_LABEL, len(subset))
    silhouette_parts.append(subset.sample(n=sample_size, random_state=RANDOM_SEED + 100 + label_value))
silhouette_df = pd.concat(silhouette_parts, ignore_index=True)
silhouette_matrix = silhouette_df[feature_cols].to_numpy(dtype=np.float32)
silhouette_labels = silhouette_df['label'].to_numpy(dtype=int)

try:
    silhouette = float(silhouette_score(silhouette_matrix, silhouette_labels, metric='euclidean'))
except Exception as exc:
    silhouette = None
    print('Silhouette score could not be computed:', exc)

separation_summary_df = pd.DataFrame(
    [
        {'metric': 'centroid_l2_distance', 'value': centroid_delta_norm},
        {'metric': 'centroid_cosine_similarity', 'value': centroid_cosine},
        {'metric': 'normal_within_class_mean_l2', 'value': float(normal_spread.mean())},
        {'metric': 'mismatch_within_class_mean_l2', 'value': float(negative_spread.mean())},
        {'metric': 'normal_projection_mean', 'value': float(projection_df.loc[projection_df['label'] == 0, 'projection_score'].mean())},
        {'metric': 'mismatch_projection_mean', 'value': float(projection_df.loc[projection_df['label'] == 1, 'projection_score'].mean())},
        {'metric': 'silhouette_score_sampled', 'value': silhouette},
    ]
)

print('Projection plot path :', projection_plot_path)
print('Silhouette sample    :', silhouette_df.shape)
print('Separation summary:')
display(separation_summary_df)

print('Projection quantiles by label:')
display(
    projection_df.groupby('label')['projection_score']
    .quantile([0.05, 0.25, 0.5, 0.75, 0.95])
    .unstack()
    .reset_index()
)


## Cell 9. Build fixed-normal ratio samples


In [ ]:
def required_negative_count(config, normal_count=NORMAL_SAMPLE_SIZE):
    return int(round(normal_count * config['negative_parts'] / config['normal_parts']))


def make_pair_key(df):
    return df['user_prompt'].astype(str) + '/x1f' + df['tool_call_text'].astype(str)


def build_ratio_sample_frames(df):
    normal_all = df[df['label'] == 0].copy()
    negative_all = df[df['label'] == 1].copy()

    base_normal_df = normal_all.sample(n=NORMAL_SAMPLE_SIZE, random_state=RANDOM_SEED).reset_index(drop=True)
    selected_groups = set(base_normal_df[SPLIT_GROUP_COL].astype(str))

    negative_candidates = negative_all[
        negative_all[SPLIT_GROUP_COL].astype(str).isin(selected_groups)
    ].copy()
    if len(negative_candidates) < MAX_NEGATIVE_SAMPLE_SIZE:
        raise ValueError(
            f'Not enough negative candidates from selected normal groups: '
            f'{len(negative_candidates)} < {MAX_NEGATIVE_SAMPLE_SIZE}'
        )

    negative_pool_df = (
        negative_candidates
        .sample(n=MAX_NEGATIVE_SAMPLE_SIZE, random_state=RANDOM_SEED + 1000)
        .reset_index(drop=True)
    )

    raw_output_cols = [
        col
        for col in [
            'user_prompt',
            'tool_call_text',
            'label',
            'prompt_source_file',
            'prompt_source_toolkit',
            'tool_source_file',
            'tool_source_toolkit',
            'source_prompt_hash',
            'normal_tool_call_text',
            'mismatch_sampling_type',
        ]
        if col in df.columns
    ]

    sample_frames = {}
    plan_rows = []
    normal_pair_keys = set(make_pair_key(base_normal_df))

    for idx, config in enumerate(RATIO_CONFIGS):
        negative_count = required_negative_count(config)
        ratio_negative_df = negative_pool_df.head(negative_count).copy()
        ratio_df = pd.concat([base_normal_df.copy(), ratio_negative_df], ignore_index=True)
        ratio_df = ratio_df.sample(frac=1, random_state=RANDOM_SEED + idx).reset_index(drop=True)

        negative_pair_keys = set(make_pair_key(ratio_negative_df))
        pair_overlap_count = len(normal_pair_keys & negative_pair_keys)
        sample_path = RUN_ROOT / f"sampled_dataset_ratio_{config['ratio_name']}.csv"
        ratio_df[raw_output_cols].to_csv(sample_path, index=False, encoding='utf-8-sig')

        label_counts = ratio_df['label'].value_counts().sort_index().to_dict()
        plan_rows.append(
            {
                'ratio_name': config['ratio_name'],
                'ratio_label': config['ratio_label'],
                'normal_count': int(label_counts.get(0, 0)),
                'negative_count': int(label_counts.get(1, 0)),
                'total_count': int(len(ratio_df)),
                'positive_rate': float(label_counts.get(1, 0) / len(ratio_df)),
                'actual_normal_to_negative': float(label_counts.get(0, 0) / max(label_counts.get(1, 0), 1)),
                'unique_source_prompt_hash': int(ratio_df[SPLIT_GROUP_COL].nunique()),
                'normal_negative_pair_overlap': int(pair_overlap_count),
                'sample_path': str(sample_path),
            }
        )
        sample_frames[config['ratio_name']] = ratio_df

    return sample_frames, pd.DataFrame(plan_rows)


ratio_sample_frames, ratio_sampling_plan_df = build_ratio_sample_frames(feature_df)
ratio_sampling_plan_path = RUN_ROOT / 'ratio_sampling_plan.csv'
ratio_sampling_plan_df.to_csv(ratio_sampling_plan_path, index=False, encoding='utf-8-sig')

print('Ratio sampling plan path:', ratio_sampling_plan_path)
display(ratio_sampling_plan_df)


## Cell 10. Train and evaluate models for each ratio


In [ ]:
ratio_experiment_results = {}
ratio_latency_artifacts = {}
ratio_split_rows = []

for ratio_index, config in enumerate(RATIO_CONFIGS):
    ratio_name = config['ratio_name']
    ratio_label = config['ratio_label']
    ratio_df = ratio_sample_frames[ratio_name]
    ratio_run_root = RUN_ROOT / f"ratio_{ratio_name}"
    ratio_run_root.mkdir(parents=True, exist_ok=True)

    print('=' * 100)
    print(f'Ratio experiment: {ratio_label} ({ratio_name})')
    print('Dataset rows:', len(ratio_df))
    print('Label counts:', ratio_df['label'].value_counts().sort_index().to_dict())

    train_df, valid_df, test_df, split_seed = group_train_valid_test_split(
        ratio_df,
        group_col=SPLIT_GROUP_COL,
        random_seed=RANDOM_SEED + ratio_index * 100,
        max_attempts=100,
    )

    print('Split seed used    :', split_seed)
    print('Split group column:', SPLIT_GROUP_COL)
    print_split_summary('train', train_df)
    print_split_summary('valid', valid_df)
    print_split_summary('test', test_df)

    for split_name, split_df in [('train', train_df), ('valid', valid_df), ('test', test_df)]:
        split_counts = split_df['label'].value_counts().sort_index().to_dict()
        ratio_split_rows.append(
            {
                'ratio_name': ratio_name,
                'ratio_label': ratio_label,
                'split': split_name,
                'rows': int(len(split_df)),
                'normal_count': int(split_counts.get(0, 0)),
                'negative_count': int(split_counts.get(1, 0)),
                'positive_rate': float(split_counts.get(1, 0) / len(split_df)),
                'unique_source_prompt_hash': int(split_df[SPLIT_GROUP_COL].nunique()),
                'split_seed': int(split_seed),
            }
        )

    ratio_experiment_results[ratio_name] = {}
    ratio_latency_artifacts[ratio_name] = {}

    for model_name in MODEL_ORDER:
        print('-' * 100)
        print(f'Running model: {model_name} | ratio={ratio_label}')

        if model_name == 'isolation_forest':
            result = run_isolation_forest_experiment(
                train_df=train_df,
                valid_df=valid_df,
                test_df=test_df,
                feature_cols=feature_cols,
                run_root=ratio_run_root,
            )
        else:
            result = run_supervised_experiment(
                model_name=model_name,
                train_df=train_df,
                valid_df=valid_df,
                test_df=test_df,
                feature_cols=feature_cols,
                run_root=ratio_run_root,
            )

        ratio_experiment_results[ratio_name][model_name] = result
        print('Best threshold :', result['summary']['best_validation_threshold'])
        print('Precision      :', result['summary']['precision'])
        print('Recall         :', result['summary']['recall'])
        print('F1             :', result['summary']['f1'])
        print('Accuracy       :', result['summary']['accuracy'])
        print('AUROC          :', result['summary']['auroc'])
        print('AUPRC          :', result['summary']['auprc'])
        print('Model device   :', result['summary']['model_device_type'])
        print('Output dir     :', result['summary']['output_dir'])

        if RUN_LATENCY_BENCHMARK:
            latency_samples_df, latency_summary, latency_report_df = benchmark_latency(result)
            ratio_latency_artifacts[ratio_name][model_name] = {
                'samples': latency_samples_df,
                'summary': latency_summary,
                'report': latency_report_df,
            }
            print('Latency model inference mean (ms):', latency_summary['model_inference_ms_mean'])

ratio_split_summary_df = pd.DataFrame(ratio_split_rows)
ratio_split_summary_path = RUN_ROOT / 'ratio_split_summary.csv'
ratio_split_summary_df.to_csv(ratio_split_summary_path, index=False, encoding='utf-8-sig')

print('Ratio split summary path:', ratio_split_summary_path)
display(ratio_split_summary_df)


## Cell 11. Aggregate ratio-study tables


In [ ]:
metric_rows = []
latency_rows = []

sampling_info = ratio_sampling_plan_df.set_index('ratio_name').to_dict(orient='index')
split_info = (
    ratio_split_summary_df[ratio_split_summary_df['split'] == 'test']
    .set_index('ratio_name')
    .to_dict(orient='index')
)

for config in RATIO_CONFIGS:
    ratio_name = config['ratio_name']
    ratio_label = config['ratio_label']
    for model_name in MODEL_ORDER:
        result = ratio_experiment_results[ratio_name][model_name]
        summary = result['summary']
        sample_summary = sampling_info[ratio_name]
        test_split_summary = split_info[ratio_name]

        metric_rows.append(
            {
                'ratio_name': ratio_name,
                'ratio_label': ratio_label,
                'model_name': model_name,
                'normal_count': int(sample_summary['normal_count']),
                'negative_count': int(sample_summary['negative_count']),
                'dataset_positive_rate': float(sample_summary['positive_rate']),
                'test_normal_count': int(test_split_summary['normal_count']),
                'test_negative_count': int(test_split_summary['negative_count']),
                'test_positive_rate': float(test_split_summary['positive_rate']),
                'score_kind': summary['score_kind'],
                'best_validation_threshold': summary['best_validation_threshold'],
                'precision': summary['precision'],
                'recall': summary['recall'],
                'f1': summary['f1'],
                'accuracy': summary['accuracy'],
                'auroc': summary['auroc'],
                'auprc': summary['auprc'],
                'tn': summary['tn'],
                'fp': summary['fp'],
                'fn': summary['fn'],
                'tp': summary['tp'],
                'fit_seconds': summary['fit_seconds'],
                'embedding_device': summary['embedding_device'],
                'model_device_type': summary['model_device_type'],
                'output_dir': summary['output_dir'],
            }
        )

        if RUN_LATENCY_BENCHMARK and model_name in ratio_latency_artifacts[ratio_name]:
            latency_summary = ratio_latency_artifacts[ratio_name][model_name]['summary']
            latency_rows.append(
                {
                    'ratio_name': ratio_name,
                    'ratio_label': ratio_label,
                    'model_name': model_name,
                    'embedding_ms_mean': latency_summary['embedding_ms_mean'],
                    'embedding_ms_median': latency_summary['embedding_ms_median'],
                    'embedding_ms_p95': latency_summary['embedding_ms_p95'],
                    'model_inference_ms_mean': latency_summary['model_inference_ms_mean'],
                    'model_inference_ms_median': latency_summary['model_inference_ms_median'],
                    'model_inference_ms_p95': latency_summary['model_inference_ms_p95'],
                    'end_to_end_ms_mean': latency_summary['end_to_end_ms_mean'],
                    'end_to_end_ms_median': latency_summary['end_to_end_ms_median'],
                    'end_to_end_ms_p95': latency_summary['end_to_end_ms_p95'],
                }
            )

ratio_metrics_df = pd.DataFrame(metric_rows)
ratio_latency_df = pd.DataFrame(latency_rows)

ratio_order = [cfg['ratio_name'] for cfg in RATIO_CONFIGS]
ratio_label_order = [cfg['ratio_label'] for cfg in RATIO_CONFIGS]
ratio_metrics_df['ratio_name'] = pd.Categorical(ratio_metrics_df['ratio_name'], categories=ratio_order, ordered=True)
if not ratio_latency_df.empty:
    ratio_latency_df['ratio_name'] = pd.Categorical(ratio_latency_df['ratio_name'], categories=ratio_order, ordered=True)

ratio_metrics_df = ratio_metrics_df.sort_values(['ratio_name', 'model_name']).reset_index(drop=True)
ratio_latency_df = ratio_latency_df.sort_values(['ratio_name', 'model_name']).reset_index(drop=True)

ratio_metrics_path = RUN_ROOT / 'ratio_metrics_comparison.csv'
ratio_latency_path = RUN_ROOT / 'ratio_latency_comparison.csv'
ratio_metrics_df.to_csv(ratio_metrics_path, index=False, encoding='utf-8-sig')
ratio_latency_df.to_csv(ratio_latency_path, index=False, encoding='utf-8-sig')

print('Ratio metrics path :', ratio_metrics_path)
print('Ratio latency path :', ratio_latency_path)
display(ratio_metrics_df)

print('AUPRC by ratio and model:')
display(
    ratio_metrics_df
    .pivot(index='ratio_label', columns='model_name', values='auprc')
    .reindex(ratio_label_order)
)

print('AUROC by ratio and model:')
display(
    ratio_metrics_df
    .pivot(index='ratio_label', columns='model_name', values='auroc')
    .reindex(ratio_label_order)
)

print('F1 by ratio and model:')
display(
    ratio_metrics_df
    .pivot(index='ratio_label', columns='model_name', values='f1')
    .reindex(ratio_label_order)
)


## Cell 12. Plot ratio-study curves


In [ ]:
ratio_plot_path = RUN_ROOT / 'ratio_study_metric_curves.png'
metrics_to_plot = [
    ('auroc', 'AUROC'),
    ('auprc', 'AUPRC'),
    ('f1', 'F1-score'),
    ('recall', 'Recall'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
axes = axes.ravel()
x = np.arange(len(ratio_label_order))

for ax, (metric_name, metric_label) in zip(axes, metrics_to_plot):
    for model_name in MODEL_ORDER:
        subset = (
            ratio_metrics_df[ratio_metrics_df['model_name'] == model_name]
            .set_index('ratio_label')
            .reindex(ratio_label_order)
        )
        ax.plot(x, subset[metric_name].to_numpy(dtype=float), marker='o', linewidth=2, label=model_name)
    ax.set_title(metric_label)
    ax.set_xticks(x)
    ax.set_xticklabels(ratio_label_order)
    ax.set_xlabel('Normal:Mismatch Ratio')
    ax.set_ylabel(metric_label)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)

fig.tight_layout()
fig.savefig(ratio_plot_path, dpi=200, bbox_inches='tight')
plt.show()

positive_rate_plot_path = RUN_ROOT / 'ratio_study_positive_rate_baseline.png'
fig, ax = plt.subplots(figsize=(8, 5))
baseline_df = (
    ratio_metrics_df[['ratio_label', 'dataset_positive_rate', 'test_positive_rate']]
    .drop_duplicates()
    .set_index('ratio_label')
    .reindex(ratio_label_order)
)
ax.plot(x, baseline_df['dataset_positive_rate'], marker='o', label='dataset positive rate')
ax.plot(x, baseline_df['test_positive_rate'], marker='s', label='test positive rate')
ax.set_xticks(x)
ax.set_xticklabels(ratio_label_order)
ax.set_xlabel('Normal:Mismatch Ratio')
ax.set_ylabel('positive rate')
ax.set_title('AUPRC baseline changes with class prevalence')
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(positive_rate_plot_path, dpi=200, bbox_inches='tight')
plt.show()

print('Ratio metric plot path       :', ratio_plot_path)
print('Positive-rate baseline path  :', positive_rate_plot_path)


## Cell 13. Output paths


In [ ]:
print('Run root:', RUN_ROOT)
print('Embedding device (used to build delta features):', EMBEDDING_DEVICE)
print('Model devices: LightGBM=', LIGHTGBM_DEVICE_CANDIDATES, ', XGBoost=', XGBOOST_DEVICE_CANDIDATES)
print()

print('Main outputs:')
print('  sampling plan:', RUN_ROOT / 'ratio_sampling_plan.csv')
print('  split summary:', RUN_ROOT / 'ratio_split_summary.csv')
print('  metrics      :', RUN_ROOT / 'ratio_metrics_comparison.csv')
print('  latency      :', RUN_ROOT / 'ratio_latency_comparison.csv')
print('  metric plot  :', RUN_ROOT / 'ratio_study_metric_curves.png')
print()

for config in RATIO_CONFIGS:
    ratio_name = config['ratio_name']
    ratio_label = config['ratio_label']
    print(f'[{ratio_label}] ratio_{ratio_name}')
    print('  sampled dataset:', RUN_ROOT / f"sampled_dataset_ratio_{ratio_name}.csv")
    for model_name in MODEL_ORDER:
        result = ratio_experiment_results[ratio_name][model_name]
        print(f'  {model_name}:', result['summary']['output_dir'])
    print()


## Cell 14. Export mismatch-labeled individual figures

This cell reuses the completed experiment outputs and saves paper-ready figures using Normal/Mismatch terminology.


In [ ]:
# Paste this whole file as one notebook cell after the ratio-study experiment has finished.
# It exports paper-ready figures with "Mismatch" terminology.

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score


INDIVIDUAL_FIG_DIR = RUN_ROOT / "paper_mismatch_individual_figures"
INDIVIDUAL_FIG_DIR.mkdir(parents=True, exist_ok=True)

RATIO_ORDER_MISMATCH = ["1:1", "7:3", "8:2", "10:1", "100:1"]
MODEL_ORDER_MISMATCH = ["lightgbm", "xgboost", "random_forest", "isolation_forest"]
MODEL_LABELS_MISMATCH = {
    "lightgbm": "LightGBM",
    "xgboost": "XGBoost",
    "random_forest": "Random Forest",
    "isolation_forest": "Isolation Forest",
}
MODEL_COLORS_MISMATCH = {
    "lightgbm": "#1f77b4",
    "xgboost": "#ff7f0e",
    "random_forest": "#2ca02c",
    "isolation_forest": "#d62728",
}

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def save_current_figure(stem):
    png_path = INDIVIDUAL_FIG_DIR / f"{stem}.png"
    svg_path = INDIVIDUAL_FIG_DIR / f"{stem}.svg"
    plt.savefig(png_path, bbox_inches="tight")
    plt.savefig(svg_path, bbox_inches="tight")
    print("Saved:", png_path)
    print("Saved:", svg_path)
    return png_path, svg_path


# Load metric results if this cell is run in a fresh kernel after the experiment.
if "ratio_metrics_df" not in globals():
    ratio_metrics_df = pd.read_csv(RUN_ROOT / "ratio_metrics_comparison.csv")

for col in ["precision", "recall", "f1", "accuracy", "auroc", "auprc"]:
    ratio_metrics_df[col] = pd.to_numeric(ratio_metrics_df[col], errors="coerce")

ratio_metrics_df["ratio_label"] = pd.Categorical(
    ratio_metrics_df["ratio_label"],
    categories=RATIO_ORDER_MISMATCH,
    ordered=True,
)

# 1. Save each ratio-study metric curve separately.
metric_specs = [
    ("auroc", "AUROC"),
    ("auprc", "AUPRC"),
    ("f1", "F1-score"),
    ("recall", "Recall"),
]
x_positions = np.arange(len(RATIO_ORDER_MISMATCH))

for metric_col, metric_label in metric_specs:
    fig, ax = plt.subplots(figsize=(7.2, 4.8))

    for model_name in MODEL_ORDER_MISMATCH:
        subset = (
            ratio_metrics_df[ratio_metrics_df["model_name"] == model_name]
            .set_index("ratio_label")
            .reindex(RATIO_ORDER_MISMATCH)
        )
        ax.plot(
            x_positions,
            subset[metric_col].to_numpy(dtype=float),
            marker="o",
            linewidth=2,
            markersize=5,
            color=MODEL_COLORS_MISMATCH[model_name],
            label=MODEL_LABELS_MISMATCH[model_name],
        )

    ax.set_title(metric_label)
    ax.set_xlabel("Normal:Mismatch Ratio")
    ax.set_ylabel(metric_label)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(RATIO_ORDER_MISMATCH)
    ax.set_ylim(0.0, 1.03)
    ax.legend(loc="best")
    fig.tight_layout()

    save_current_figure(f"mismatch_metric_curve_{metric_col}")
    plt.show()


# 2. Delta embedding L2 norm, PCA, and centroid-projection figures.
if "feature_df" not in globals() or "feature_cols" not in globals():
    raise RuntimeError("feature_df and feature_cols are required. Run the delta feature generation cells first.")

ANALYSIS_SAMPLE_PER_LABEL_LOCAL = globals().get("ANALYSIS_SAMPLE_PER_LABEL", 10000)
SILHOUETTE_SAMPLE_PER_LABEL_LOCAL = globals().get("SILHOUETTE_SAMPLE_PER_LABEL", 1000)

analysis_parts = []
for label_value in [0, 1]:
    subset = feature_df[feature_df["label"] == label_value]
    sample_size = min(ANALYSIS_SAMPLE_PER_LABEL_LOCAL, len(subset))
    analysis_parts.append(subset.sample(n=sample_size, random_state=RANDOM_SEED + label_value))

analysis_df = (
    pd.concat(analysis_parts, ignore_index=True)
    .sample(frac=1, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

analysis_delta_matrix = analysis_df[feature_cols].to_numpy(dtype=np.float32)
analysis_labels = analysis_df["label"].to_numpy(dtype=int)
analysis_delta_norms = np.linalg.norm(analysis_delta_matrix, axis=1).astype(np.float32)

plot_df = analysis_df[["label"]].copy()
plot_df["delta_l2_norm"] = analysis_delta_norms

# 2-1. L2 norm distribution.
fig, ax = plt.subplots(figsize=(7.2, 4.8))
for label_value, color, label_name in [
    (0, "#1f77b4", "Normal (label 0)"),
    (1, "#d62728", "Mismatch (label 1)"),
]:
    subset = plot_df[plot_df["label"] == label_value]
    ax.hist(
        subset["delta_l2_norm"],
        bins=40,
        alpha=0.55,
        density=True,
        color=color,
        label=label_name,
    )

ax.set_title("Delta Embedding L2 Norm Distribution")
ax.set_xlabel("L2 norm of prompt_emb - tool_emb")
ax.set_ylabel("Density")
ax.legend()
fig.tight_layout()
save_current_figure("mismatch_delta_l2_norm_histogram")
plt.show()

l2_summary_df = (
    plot_df.groupby("label")["delta_l2_norm"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .reset_index()
)
l2_quantile_df = (
    plot_df.groupby("label")["delta_l2_norm"]
    .quantile([0.05, 0.25, 0.5, 0.75, 0.95])
    .unstack()
    .reset_index()
)
l2_summary_path = INDIVIDUAL_FIG_DIR / "mismatch_delta_l2_norm_summary.csv"
l2_quantile_path = INDIVIDUAL_FIG_DIR / "mismatch_delta_l2_norm_quantiles.csv"
l2_summary_df.to_csv(l2_summary_path, index=False, encoding="utf-8-sig")
l2_quantile_df.to_csv(l2_quantile_path, index=False, encoding="utf-8-sig")
print("Saved:", l2_summary_path)
print("Saved:", l2_quantile_path)
display(l2_summary_df)
display(l2_quantile_df)

# 2-2. PCA scatter.
pca = PCA(n_components=2, random_state=RANDOM_SEED)
delta_pca_2d = pca.fit_transform(analysis_delta_matrix)

pca_df = plot_df.copy()
pca_df["pca_x"] = delta_pca_2d[:, 0]
pca_df["pca_y"] = delta_pca_2d[:, 1]

fig, ax = plt.subplots(figsize=(7.2, 5.4))
for label_value, color, label_name in [
    (0, "#1f77b4", "Normal (label 0)"),
    (1, "#d62728", "Mismatch (label 1)"),
]:
    subset = pca_df[pca_df["label"] == label_value]
    ax.scatter(
        subset["pca_x"],
        subset["pca_y"],
        s=12,
        alpha=0.55,
        color=color,
        label=label_name,
    )

ax.set_title("Delta Embedding PCA")
ax.set_xlabel("PCA 1")
ax.set_ylabel("PCA 2")
ax.legend()
fig.tight_layout()
save_current_figure("mismatch_delta_pca_scatter")
plt.show()

pca_csv_path = INDIVIDUAL_FIG_DIR / "mismatch_delta_pca_sample_coordinates.csv"
pca_summary_path = INDIVIDUAL_FIG_DIR / "mismatch_delta_pca_summary.csv"
pca_df.to_csv(pca_csv_path, index=False, encoding="utf-8-sig")
pca_summary_df = pd.DataFrame(
    {
        "component": ["PCA1", "PCA2"],
        "explained_variance_ratio": pca.explained_variance_ratio_,
    }
)
pca_summary_df.to_csv(pca_summary_path, index=False, encoding="utf-8-sig")
print("Saved:", pca_csv_path)
print("Saved:", pca_summary_path)
display(pca_summary_df)

# 2-3. Projection on centroid-difference axis.
normal_matrix = analysis_delta_matrix[analysis_labels == 0]
mismatch_matrix = analysis_delta_matrix[analysis_labels == 1]

normal_centroid = normal_matrix.mean(axis=0)
mismatch_centroid = mismatch_matrix.mean(axis=0)
centroid_delta = mismatch_centroid - normal_centroid
centroid_delta_norm = float(np.linalg.norm(centroid_delta))

normal_centroid_norm = float(np.linalg.norm(normal_centroid))
mismatch_centroid_norm = float(np.linalg.norm(mismatch_centroid))
centroid_cosine = float(
    np.dot(normal_centroid, mismatch_centroid)
    / max(normal_centroid_norm * mismatch_centroid_norm, 1e-12)
)

normal_spread = np.linalg.norm(normal_matrix - normal_centroid, axis=1)
mismatch_spread = np.linalg.norm(mismatch_matrix - mismatch_centroid, axis=1)

if centroid_delta_norm > 0:
    separation_axis = centroid_delta / centroid_delta_norm
else:
    separation_axis = np.zeros_like(centroid_delta)

projection_scores = analysis_delta_matrix @ separation_axis
projection_df = pd.DataFrame(
    {
        "label": analysis_labels,
        "projection_score": projection_scores,
    }
)

fig, ax = plt.subplots(figsize=(7.2, 4.8))
for label_value, color, label_name in [
    (0, "#1f77b4", "Normal (label 0)"),
    (1, "#d62728", "Mismatch (label 1)"),
]:
    subset = projection_df[projection_df["label"] == label_value]
    ax.hist(
        subset["projection_score"],
        bins=40,
        alpha=0.55,
        density=True,
        color=color,
        label=label_name,
    )

ax.set_title("Projection on Centroid-Difference Axis")
ax.set_xlabel("Projection score")
ax.set_ylabel("Density")
ax.legend()
fig.tight_layout()
save_current_figure("mismatch_delta_centroid_projection_histogram")
plt.show()

projection_csv_path = INDIVIDUAL_FIG_DIR / "mismatch_delta_centroid_projection_scores.csv"
projection_quantile_path = INDIVIDUAL_FIG_DIR / "mismatch_delta_centroid_projection_quantiles.csv"
separation_summary_path = INDIVIDUAL_FIG_DIR / "mismatch_delta_separation_summary.csv"
projection_df.to_csv(projection_csv_path, index=False, encoding="utf-8-sig")

projection_quantile_df = (
    projection_df.groupby("label")["projection_score"]
    .quantile([0.05, 0.25, 0.5, 0.75, 0.95])
    .unstack()
    .reset_index()
)
projection_quantile_df.to_csv(projection_quantile_path, index=False, encoding="utf-8-sig")

silhouette_parts = []
for label_value in [0, 1]:
    subset = analysis_df[analysis_df["label"] == label_value]
    sample_size = min(SILHOUETTE_SAMPLE_PER_LABEL_LOCAL, len(subset))
    silhouette_parts.append(subset.sample(n=sample_size, random_state=RANDOM_SEED + 100 + label_value))

silhouette_df = pd.concat(silhouette_parts, ignore_index=True)
silhouette_matrix = silhouette_df[feature_cols].to_numpy(dtype=np.float32)
silhouette_labels = silhouette_df["label"].to_numpy(dtype=int)

try:
    silhouette = float(silhouette_score(silhouette_matrix, silhouette_labels, metric="euclidean"))
except Exception as exc:
    silhouette = None
    print("Silhouette score could not be computed:", exc)

separation_summary_df = pd.DataFrame(
    [
        {"metric": "centroid_l2_distance", "value": centroid_delta_norm},
        {"metric": "centroid_cosine_similarity", "value": centroid_cosine},
        {"metric": "normal_within_class_mean_l2", "value": float(normal_spread.mean())},
        {"metric": "mismatch_within_class_mean_l2", "value": float(mismatch_spread.mean())},
        {
            "metric": "normal_projection_mean",
            "value": float(projection_df.loc[projection_df["label"] == 0, "projection_score"].mean()),
        },
        {
            "metric": "mismatch_projection_mean",
            "value": float(projection_df.loc[projection_df["label"] == 1, "projection_score"].mean()),
        },
        {"metric": "silhouette_score_sampled", "value": silhouette},
    ]
)
separation_summary_df.to_csv(separation_summary_path, index=False, encoding="utf-8-sig")
print("Saved:", projection_csv_path)
print("Saved:", projection_quantile_path)
print("Saved:", separation_summary_path)
display(projection_quantile_df)
display(separation_summary_df)

print()
print("All mismatch-labeled outputs saved to:", INDIVIDUAL_FIG_DIR)
